In [ ]:
import os
import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()

from detectron2 import model_zoo
from detectron2.engine import DefaultTrainer, DefaultPredictor
from detectron2.config import get_cfg
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.data.datasets import register_coco_instances
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader

def setup_dataset():
    """데이터셋 등록 - train, val, test"""
    # 기존 데이터셋 등록 제거
    if "my_train_dataset" in DatasetCatalog.list():
        DatasetCatalog.remove("my_train_dataset")
    if "my_val_dataset" in DatasetCatalog.list():
        DatasetCatalog.remove("my_val_dataset")
    if "my_test_dataset" in DatasetCatalog.list():
        DatasetCatalog.remove("my_test_dataset")
    
    # 데이터셋 새로 등록
    register_coco_instances(
        "my_train_dataset", 
        {}, 
        "C:/Users/AI-LHJ/Desktop/coco-segmentation/train/_annotations.coco.json",
        "C:/Users/AI-LHJ/Desktop/coco-segmentation/train"
    )
    
    register_coco_instances(
        "my_val_dataset", 
        {}, 
        "C:/Users/AI-LHJ/Desktop/coco-segmentation/valid/_annotations.coco.json",
        "C:/Users/AI-LHJ/Desktop/coco-segmentation/valid"
    )
    
    register_coco_instances(
        "my_test_dataset", 
        {}, 
        "C:/Users/AI-LHJ/Desktop/coco-segmentation/test/_annotations.coco.json",
        "C:/Users/AI-LHJ/Desktop/coco-segmentation/test"
    )

class CustomTrainer(DefaultTrainer):
    """커스텀 트레이너 - 검증 데이터셋에 대한 평가 기능 추가"""
    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        if output_folder is None:
            output_folder = os.path.join(cfg.OUTPUT_DIR, "inference")
        return COCOEvaluator(dataset_name, cfg, True, output_folder)

def setup_config():
    """모델 설정"""
    cfg = get_cfg()
    
    # 기본 설정
    cfg.MODEL.META_ARCHITECTURE = "GeneralizedRCNN"
    cfg.MODEL.BACKBONE.NAME = "build_resnet_fpn_backbone"
    cfg.MODEL.RESNETS.OUT_FEATURES = ["res2", "res3", "res4", "res5"]
    cfg.MODEL.FPN.IN_FEATURES = ["res2", "res3", "res4", "res5"]
    
    # ROI_BOX_HEAD 설정 추가
    cfg.MODEL.ROI_HEADS.NAME = "StandardROIHeads"
    cfg.MODEL.ROI_BOX_HEAD.NAME = "FastRCNNConvFCHead"  # 이 줄 추가
    cfg.MODEL.ROI_BOX_HEAD.NUM_FC = 2  # 이 줄 추가
    cfg.MODEL.ROI_BOX_HEAD.POOLER_RESOLUTION = 7  # 이 줄 추가
    
    cfg.MODEL.ROI_HEADS.IN_FEATURES = ["p2", "p3", "p4", "p5"]
    cfg.MODEL.ANCHOR_GENERATOR.SIZES = [[32], [64], [128], [256], [512]]
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [[0.5, 1.0, 2.0]]
    cfg.MODEL.RPN.IN_FEATURES = ["p2", "p3", "p4", "p5", "p6"]
    cfg.MODEL.RPN.PRE_NMS_TOPK_TRAIN = 2000
    cfg.MODEL.RPN.PRE_NMS_TOPK_TEST = 1000
    cfg.MODEL.RPN.POST_NMS_TOPK_TRAIN = 1000
    cfg.MODEL.RPN.POST_NMS_TOPK_TEST = 1000
    cfg.MODEL.ROI_MASK_HEAD.NAME = "MaskRCNNConvUpsampleHead"
    cfg.MODEL.ROI_MASK_HEAD.NUM_CONV = 4
    cfg.MODEL.ROI_MASK_HEAD.POOLER_RESOLUTION = 14
    
    # 나머지 설정은 동일...
    cfg.DATASETS.TRAIN = ("my_train_dataset",)
    cfg.DATASETS.TEST = ("my_val_dataset", "my_test_dataset")
    
    cfg.MODEL.WEIGHTS = "detectron2://COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x/137849600/model_final_f10217.pkl"
    
    cfg.DATALOADER.NUM_WORKERS = 8
    cfg.SOLVER.IMS_PER_BATCH = 8
    cfg.SOLVER.BASE_LR = 0.001
    cfg.SOLVER.MAX_ITER = 1000
    cfg.SOLVER.STEPS = (300, 800)  # 이 지점에서 학습률을 감소
    cfg.SOLVER.GAMMA = 0.1  # 학습률 감소 비율
    cfg.SOLVER.WARMUP_FACTOR = 1.0 / 1000
    cfg.SOLVER.WARMUP_ITERS = 200
    cfg.SOLVER.WARMUP_METHOD = "linear"
    cfg.TEST.EVAL_PERIOD = 100
    cfg.MODEL.ROI_HEADS.NUM_CLASSES = 4
    
    cfg.OUTPUT_DIR = "./output"
    os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
    
    return cfg

def train_model():
    """모델 학습"""
    # 데이터셋 설정
    setup_dataset()
    
    # 설정 가져오기
    cfg = setup_config()
    
    # 트레이너 생성 및 학습 시작
    trainer = CustomTrainer(cfg)
    trainer.resume_or_load(resume=False)
    trainer.train()
    
    return cfg, trainer

def evaluate_model(cfg, trainer):
    """학습된 모델 평가"""
    # 검증 데이터셋 평가
    evaluator_val = COCOEvaluator(
        "my_val_dataset", 
        cfg, 
        False, 
        output_dir=os.path.join(cfg.OUTPUT_DIR, "validation")
    )
    val_loader = build_detection_test_loader(cfg, "my_val_dataset")
    inference_on_dataset(trainer.model, val_loader, evaluator_val)
    
    # 테스트 데이터셋 평가
    evaluator_test = COCOEvaluator(
        "my_test_dataset", 
        cfg, 
        False, 
        output_dir=os.path.join(cfg.OUTPUT_DIR, "test")
    )
    test_loader = build_detection_test_loader(cfg, "my_test_dataset")
    inference_on_dataset(trainer.model, test_loader, evaluator_test)

def main():
    """메인 함수"""
    # 학습 실행
    cfg, trainer = train_model()
    
    print("Training completed!")
    
    # 평가 실행
    print("Starting evaluation...")
    evaluate_model(cfg, trainer)
    print("Evaluation completed!")

if __name__ == "__main__":
    main()

In [ ]:
import os
import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()

from detectron2 import model_zoo
from detectron2.engine import DefaultTrainer, DefaultPredictor
from detectron2.config import get_cfg
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.data.datasets import register_coco_instances
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader

def setup_dataset():
    """데이터셋 등록 - train, val, test"""
    # 기존 데이터셋 등록 제거
    if "my_train_dataset" in DatasetCatalog.list():
        DatasetCatalog.remove("my_train_dataset")
    if "my_val_dataset" in DatasetCatalog.list():
        DatasetCatalog.remove("my_val_dataset")
    if "my_test_dataset" in DatasetCatalog.list():
        DatasetCatalog.remove("my_test_dataset")
    
    # 데이터셋 새로 등록
    register_coco_instances(
        "my_train_dataset", 
        {}, 
        "C:/Users/AI-LHJ/Desktop/coco-segmentation/train/_annotations.coco.json",
        "C:/Users/AI-LHJ/Desktop/coco-segmentation/train"
    )
    
    register_coco_instances(
        "my_val_dataset", 
        {}, 
        "C:/Users/AI-LHJ/Desktop/coco-segmentation/valid/_annotations.coco.json",
        "C:/Users/AI-LHJ/Desktop/coco-segmentation/valid"
    )
    
    register_coco_instances(
        "my_test_dataset", 
        {}, 
        "C:/Users/AI-LHJ/Desktop/coco-segmentation/test/_annotations.coco.json",
        "C:/Users/AI-LHJ/Desktop/coco-segmentation/test"
    )

class CustomTrainer(DefaultTrainer):
    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        if output_folder is None:
            output_folder = os.path.join(cfg.OUTPUT_DIR, "inference")
        return COCOEvaluator(dataset_name, tasks=("bbox",), distributed=False, output_dir=output_folder)

def setup_config():
    cfg = get_cfg()
    
    # 기본 설정
    cfg.MODEL.META_ARCHITECTURE = "GeneralizedRCNN"
    cfg.MODEL.BACKBONE.NAME = "build_resnet_fpn_backbone"
    cfg.MODEL.RESNETS.OUT_FEATURES = ["res2", "res3", "res4", "res5"]
    cfg.MODEL.FPN.IN_FEATURES = ["res2", "res3", "res4", "res5"]
    
    cfg.MODEL.ROI_HEADS.NAME = "StandardROIHeads"
    cfg.MODEL.ROI_BOX_HEAD.NAME = "FastRCNNConvFCHead"
    cfg.MODEL.ROI_BOX_HEAD.NUM_FC = 2
    cfg.MODEL.ROI_BOX_HEAD.POOLER_RESOLUTION = 7
    
    cfg.MODEL.ROI_HEADS.IN_FEATURES = ["p2", "p3", "p4", "p5"]
    cfg.MODEL.ANCHOR_GENERATOR.SIZES = [[32], [64], [128], [256], [512]]
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [[0.5, 1.0, 2.0]]
    cfg.MODEL.RPN.IN_FEATURES = ["p2", "p3", "p4", "p5", "p6"]
    cfg.MODEL.RPN.PRE_NMS_TOPK_TRAIN = 2000
    cfg.MODEL.RPN.PRE_NMS_TOPK_TEST = 1000
    cfg.MODEL.RPN.POST_NMS_TOPK_TRAIN = 1000
    cfg.MODEL.RPN.POST_NMS_TOPK_TEST = 1000
    
    cfg.DATASETS.TRAIN = ("my_train_dataset",)
    cfg.DATASETS.TEST = ("my_val_dataset",)
    
    cfg.MODEL.WEIGHTS = "detectron2://COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x/137849600/model_final_f10217.pkl"
    
    cfg.SOLVER.IMS_PER_BATCH = 8
    cfg.SOLVER.BASE_LR = 0.0005
    cfg.SOLVER.MAX_ITER = 4000
    cfg.SOLVER.STEPS = (1000, 2000, 3000)
    cfg.SOLVER.GAMMA = 0.5
    
    cfg.MODEL.ROI_HEADS.NUM_CLASSES = 4
    
    cfg.OUTPUT_DIR = "./output"
    os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
    
    return cfg

def train_model():
    setup_dataset()
    cfg = setup_config()
    trainer = CustomTrainer(cfg)
    trainer.resume_or_load(resume=False)
    trainer.train()
    return cfg, trainer

def evaluate_model(cfg, trainer):
    """학습된 모델 평가"""
    print("Starting evaluation...")
    
    evaluator = COCOEvaluator(
        "my_val_dataset", 
        tasks=("bbox",),
        distributed=False,
        output_dir=os.path.join(cfg.OUTPUT_DIR, "validation")
    )
    
    val_loader = build_detection_test_loader(cfg, "my_val_dataset")
    results = inference_on_dataset(trainer.model, val_loader, evaluator)
    
    # 전체 메트릭
    print("\nOverall Metrics:")
    print(f"AP50 (mAP@0.5): {results['bbox']['AP50']:.3f}")
    print(f"AP75 (mAP@0.75): {results['bbox']['AP75']:.3f}")
    print(f"AP (mAP@[0.5:0.95]): {results['bbox']['AP']:.3f}")
    
    # 클래스별 AP 출력
    print("\nPer-category bbox AP:")
    categories = MetadataCatalog.get("my_val_dataset").thing_classes
    
    # COCO 평가기의 결과에서 클래스별 AP 추출
    if 'segm' in results:
        metrics = results['segm']
    else:
        metrics = results['bbox']
    
    precisions = metrics.get('precision', [])
    if len(precisions) > 0:
        # 각 클래스별로 정밀도 계산
        for idx, cat in enumerate(categories):
            if idx < len(precisions):
                ap = float(precisions[idx].mean())
                ap50 = float(precisions[idx, :, :, 0, -1].mean())
                ap75 = float(precisions[idx, :, :, 1, -1].mean())
                print(f"{cat:<20}: AP={ap:.3f}, AP50={ap50:.3f}, AP75={ap75:.3f}")

def main():
    cfg, trainer = train_model()
    print("Training completed!")
    evaluate_model(cfg, trainer)
    print("Evaluation completed!")

if __name__ == "__main__":
    main()

In [ ]:
import os
import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()

from detectron2 import model_zoo
from detectron2.engine import DefaultTrainer, DefaultPredictor
from detectron2.config import get_cfg
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.data.datasets import register_coco_instances
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader

def setup_dataset():
    """데이터셋 등록 - train, val, test"""
    # 기존 데이터셋 등록 제거
    if "my_train_dataset" in DatasetCatalog.list():
        DatasetCatalog.remove("my_train_dataset")
    if "my_val_dataset" in DatasetCatalog.list():
        DatasetCatalog.remove("my_val_dataset")
    if "my_test_dataset" in DatasetCatalog.list():
        DatasetCatalog.remove("my_test_dataset")
    
    # 데이터셋 새로 등록
    register_coco_instances(
        "my_train_dataset", 
        {}, 
        "C:/Users/AI-LHJ/Desktop/coco-segmentation/train/_annotations.coco.json",
        "C:/Users/AI-LHJ/Desktop/coco-segmentation/train"
    )
    
    register_coco_instances(
        "my_val_dataset", 
        {}, 
        "C:/Users/AI-LHJ/Desktop/coco-segmentation/valid/_annotations.coco.json",
        "C:/Users/AI-LHJ/Desktop/coco-segmentation/valid"
    )
    
    register_coco_instances(
        "my_test_dataset", 
        {}, 
        "C:/Users/AI-LHJ/Desktop/coco-segmentation/test/_annotations.coco.json",
        "C:/Users/AI-LHJ/Desktop/coco-segmentation/test"
    )

class CustomTrainer(DefaultTrainer):
    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        if output_folder is None:
            output_folder = os.path.join(cfg.OUTPUT_DIR, "inference")
        return COCOEvaluator(dataset_name, tasks=("bbox",), distributed=False, output_dir=output_folder)

def setup_config():
    cfg = get_cfg()
    
    # 기본 설정
    cfg.MODEL.META_ARCHITECTURE = "GeneralizedRCNN"
    cfg.MODEL.BACKBONE.NAME = "build_resnet_fpn_backbone"
    cfg.MODEL.RESNETS.OUT_FEATURES = ["res2", "res3", "res4", "res5"]
    cfg.MODEL.FPN.IN_FEATURES = ["res2", "res3", "res4", "res5"]
    
    cfg.MODEL.ROI_HEADS.NAME = "StandardROIHeads"
    cfg.MODEL.ROI_BOX_HEAD.NAME = "FastRCNNConvFCHead"
    cfg.MODEL.ROI_BOX_HEAD.NUM_FC = 2
    cfg.MODEL.ROI_BOX_HEAD.POOLER_RESOLUTION = 7
    
    cfg.MODEL.ROI_HEADS.IN_FEATURES = ["p2", "p3", "p4", "p5"]
    cfg.MODEL.ANCHOR_GENERATOR.SIZES = [[32], [64], [128], [256], [512]]
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [[0.5, 1.0, 2.0]]
    cfg.MODEL.RPN.IN_FEATURES = ["p2", "p3", "p4", "p5", "p6"]
    cfg.MODEL.RPN.PRE_NMS_TOPK_TRAIN = 2000
    cfg.MODEL.RPN.PRE_NMS_TOPK_TEST = 1000
    cfg.MODEL.RPN.POST_NMS_TOPK_TRAIN = 1000
    cfg.MODEL.RPN.POST_NMS_TOPK_TEST = 1000
    
    cfg.DATASETS.TRAIN = ("my_train_dataset",)
    cfg.DATASETS.TEST = ("my_val_dataset",)
    
    cfg.MODEL.WEIGHTS = "detectron2://COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x/137849600/model_final_f10217.pkl"
    
    cfg.SOLVER.IMS_PER_BATCH = 8
    cfg.SOLVER.BASE_LR = 0.0005
    cfg.SOLVER.MAX_ITER = 4000
    cfg.SOLVER.STEPS = (1000, 2000, 3000)
    cfg.SOLVER.GAMMA = 0.5
    
    cfg.MODEL.ROI_HEADS.NUM_CLASSES = 4
    
    cfg.OUTPUT_DIR = "./output"
    os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
    
    return cfg

def train_model():
    setup_dataset()
    cfg = setup_config()
    trainer = CustomTrainer(cfg)
    trainer.resume_or_load(resume=False)
    trainer.train()
    return cfg, trainer

def evaluate_model(cfg, trainer):
    """학습된 모델 평가"""
    print("Starting evaluation...")
    
    evaluator = COCOEvaluator(
        "my_val_dataset", 
        tasks=("bbox",),
        distributed=False,
        output_dir=os.path.join(cfg.OUTPUT_DIR, "validation")
    )
    
    val_loader = build_detection_test_loader(cfg, "my_val_dataset")
    results = inference_on_dataset(trainer.model, val_loader, evaluator)
    
    metrics = results['bbox']
    
    # 결과를 표 형식으로 출력
    print("\nDetailed Evaluation Results:")
    print(f"{'Class':<15} {'Precision':>10} {'Recall':>10} {'mAP50':>10} {'mAP50-95':>10}")
    print("-" * 60)
    
    # 전체 성능 출력
    print(f"{'All':<15} {metrics.get('precision_total', 0):>10.3f} {metrics.get('recall_total', 0):>10.3f} {metrics.get('AP50', 0):>10.3f} {metrics.get('AP', 0):>10.3f}")
    
    # 클래스별 성능 출력
    categories = MetadataCatalog.get("my_val_dataset").thing_classes
    precisions = metrics.get('precision', [])
    recalls = metrics.get('recall', [])
    
    if len(precisions) > 0:
        for idx, cat in enumerate(categories):
            if idx < len(precisions):
                # Precision 계산 (IoU=0.5에서의 평균값)
                precision = float(precisions[idx, :, :, 0, -1].mean())
                
                # Recall 계산
                recall = float(recalls[idx, :, 0, -1].mean())
                
                # AP50 계산
                ap50 = float(precisions[idx, :, :, 0, -1].mean())
                
                # AP (mAP50-95) 계산
                ap = float(precisions[idx].mean())
                
                print(f"{cat:<15} {precision:>10.3f} {recall:>10.3f} {ap50:>10.3f} {ap:>10.3f}")

def main():
    cfg, trainer = train_model()
    print("Training completed!")
    evaluate_model(cfg, trainer)
    print("Evaluation completed!")

if __name__ == "__main__":
    main()

In [ ]:
import os
import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()

from detectron2 import model_zoo
from detectron2.engine import DefaultTrainer, DefaultPredictor
from detectron2.config import get_cfg
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.data.datasets import register_coco_instances
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader

def setup_dataset():
    """데이터셋 등록 - train, val, test"""
    # 기존 데이터셋 등록 제거
    if "my_train_dataset" in DatasetCatalog.list():
        DatasetCatalog.remove("my_train_dataset")
    if "my_val_dataset" in DatasetCatalog.list():
        DatasetCatalog.remove("my_val_dataset")
    if "my_test_dataset" in DatasetCatalog.list():
        DatasetCatalog.remove("my_test_dataset")
    
    # 데이터셋 새로 등록
    register_coco_instances(
        "my_train_dataset", 
        {}, 
        "C:/Users/AI-LHJ/Desktop/coco-segmentation/train/_annotations.coco.json",
        "C:/Users/AI-LHJ/Desktop/coco-segmentation/train"
    )
    
    register_coco_instances(
        "my_val_dataset", 
        {}, 
        "C:/Users/AI-LHJ/Desktop/coco-segmentation/valid/_annotations.coco.json",
        "C:/Users/AI-LHJ/Desktop/coco-segmentation/valid"
    )
    
    register_coco_instances(
        "my_test_dataset", 
        {}, 
        "C:/Users/AI-LHJ/Desktop/coco-segmentation/test/_annotations.coco.json",
        "C:/Users/AI-LHJ/Desktop/coco-segmentation/test"
    )

class CustomTrainer(DefaultTrainer):
    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        if output_folder is None:
            output_folder = os.path.join(cfg.OUTPUT_DIR, "inference")
        return COCOEvaluator(dataset_name, tasks=("bbox",), distributed=False, output_dir=output_folder)

def setup_config():
    cfg = get_cfg()
    
    # 기본 설정
    cfg.MODEL.META_ARCHITECTURE = "GeneralizedRCNN"
    cfg.MODEL.BACKBONE.NAME = "build_resnet_fpn_backbone"
    cfg.MODEL.RESNETS.OUT_FEATURES = ["res2", "res3", "res4", "res5"]
    cfg.MODEL.FPN.IN_FEATURES = ["res2", "res3", "res4", "res5"]
    
    cfg.MODEL.ROI_HEADS.NAME = "StandardROIHeads"
    cfg.MODEL.ROI_BOX_HEAD.NAME = "FastRCNNConvFCHead"
    cfg.MODEL.ROI_BOX_HEAD.NUM_FC = 2
    cfg.MODEL.ROI_BOX_HEAD.POOLER_RESOLUTION = 7
    
    cfg.MODEL.ROI_HEADS.IN_FEATURES = ["p2", "p3", "p4", "p5"]
    cfg.MODEL.ANCHOR_GENERATOR.SIZES = [[32], [64], [128], [256], [512]]
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [[0.5, 1.0, 2.0]]
    cfg.MODEL.RPN.IN_FEATURES = ["p2", "p3", "p4", "p5", "p6"]
    cfg.MODEL.RPN.PRE_NMS_TOPK_TRAIN = 2000
    cfg.MODEL.RPN.PRE_NMS_TOPK_TEST = 1000
    cfg.MODEL.RPN.POST_NMS_TOPK_TRAIN = 1000
    cfg.MODEL.RPN.POST_NMS_TOPK_TEST = 1000
    
    cfg.DATASETS.TRAIN = ("my_train_dataset",)
    cfg.DATASETS.TEST = ("my_val_dataset",)
    
    cfg.MODEL.WEIGHTS = "detectron2://COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x/137849600/model_final_f10217.pkl"
    
    cfg.SOLVER.IMS_PER_BATCH = 8
    cfg.SOLVER.BASE_LR = 0.0005
    cfg.SOLVER.MAX_ITER = 4000
    cfg.SOLVER.STEPS = (1000, 2000, 3000)
    cfg.SOLVER.GAMMA = 0.5
    
    cfg.MODEL.ROI_HEADS.NUM_CLASSES = 4
    
    cfg.OUTPUT_DIR = "./output"
    os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
    
    return cfg

def train_model():
    setup_dataset()
    cfg = setup_config()
    trainer = CustomTrainer(cfg)
    trainer.resume_or_load(resume=False)
    trainer.train()
    return cfg, trainer

def evaluate_model(cfg, trainer):
    """학습된 모델 평가"""
    print("Starting evaluation...")
    
    evaluator = COCOEvaluator(
        "my_val_dataset", 
        tasks=("bbox",),
        distributed=False,
        output_dir=os.path.join(cfg.OUTPUT_DIR, "validation")
    )
    
    val_loader = build_detection_test_loader(cfg, "my_val_dataset")
    results = inference_on_dataset(trainer.model, val_loader, evaluator)
    metrics = results['bbox']
    
    # 전체 성능 지표
    mAP = metrics.get('AP', 0)
    mAP50 = metrics.get('AP50', 0)
    
    # 결과를 표 형식으로 출력
    print("\nDetailed Evaluation Results:")
    print(f"{'Class':<15} {'Precision':>10} {'Recall':>10} {'mAP50':>10} {'mAP50-95':>10}")
    print("-" * 60)
    
    # COCO 평가기의 평가 결과에서 클래스별 AP 값을 가져옴
    if 'precision' in metrics and 'recall' in metrics:
        precisions = metrics['precision']
        recalls = metrics['recall']
        
        categories = ['wire', 'crossarm', 'polo', 'wire-crossarm-polo']
        
        class_metrics = []
        for idx, category in enumerate(categories):
            if idx < len(precisions):
                # 클래스별 precision (IoU=0.5에서의 값)
                precision = float(precisions[idx, :, 0, -1].mean())
                
                # 클래스별 recall
                recall = float(recalls[idx, 0, -1])
                
                # 클래스별 AP50
                ap50 = float(precisions[idx, :, 0, -1].mean())
                
                # 클래스별 AP (mAP50-95)
                ap = float(precisions[idx].mean())
                
                print(f"{category:<15} {precision:>10.3f} {recall:>10.3f} {ap50:>10.3f} {ap:>10.3f}")
                
                class_metrics.append({
                    'precision': precision,
                    'recall': recall,
                    'ap50': ap50,
                    'ap': ap
                })
        
        # 전체 평균 계산
        avg_precision = sum(m['precision'] for m in class_metrics) / len(class_metrics)
        avg_recall = sum(m['recall'] for m in class_metrics) / len(class_metrics)
        
        print("-" * 60)
        print(f"{'Total':<15} {avg_precision:>10.3f} {avg_recall:>10.3f} {mAP50:>10.3f} {mAP:>10.3f}")

def main():
    cfg, trainer = train_model()
    print("Training completed!")
    evaluate_model(cfg, trainer)
    print("Evaluation completed!")

if __name__ == "__main__":
    main()

In [ ]:
import os
import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()

from detectron2 import model_zoo
from detectron2.engine import DefaultTrainer, DefaultPredictor
from detectron2.config import get_cfg
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.data.datasets import register_coco_instances
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader

def setup_dataset():
    """데이터셋 등록 - train, val, test"""
    # 기존 데이터셋 등록 제거
    if "my_train_dataset" in DatasetCatalog.list():
        DatasetCatalog.remove("my_train_dataset")
    if "my_val_dataset" in DatasetCatalog.list():
        DatasetCatalog.remove("my_val_dataset")
    if "my_test_dataset" in DatasetCatalog.list():
        DatasetCatalog.remove("my_test_dataset")
    
    # 데이터셋 새로 등록
    register_coco_instances(
        "my_train_dataset", 
        {}, 
        "C:/Users/AI-LHJ/Desktop/coco-segmentation/train/_annotations.coco.json",
        "C:/Users/AI-LHJ/Desktop/coco-segmentation/train"
    )
    
    register_coco_instances(
        "my_val_dataset", 
        {}, 
        "C:/Users/AI-LHJ/Desktop/coco-segmentation/valid/_annotations.coco.json",
        "C:/Users/AI-LHJ/Desktop/coco-segmentation/valid"
    )
    
    register_coco_instances(
        "my_test_dataset", 
        {}, 
        "C:/Users/AI-LHJ/Desktop/coco-segmentation/test/_annotations.coco.json",
        "C:/Users/AI-LHJ/Desktop/coco-segmentation/test"
    )

class CustomTrainer(DefaultTrainer):
    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        if output_folder is None:
            output_folder = os.path.join(cfg.OUTPUT_DIR, "inference")
        return COCOEvaluator(dataset_name, tasks=("bbox",), distributed=False, output_dir=output_folder)

def setup_config():
    cfg = get_cfg()
    
    # 기본 설정
    cfg.MODEL.META_ARCHITECTURE = "GeneralizedRCNN"
    cfg.MODEL.BACKBONE.NAME = "build_resnet_fpn_backbone"
    cfg.MODEL.RESNETS.OUT_FEATURES = ["res2", "res3", "res4", "res5"]
    cfg.MODEL.FPN.IN_FEATURES = ["res2", "res3", "res4", "res5"]
    
    cfg.MODEL.ROI_HEADS.NAME = "StandardROIHeads"
    cfg.MODEL.ROI_BOX_HEAD.NAME = "FastRCNNConvFCHead"
    cfg.MODEL.ROI_BOX_HEAD.NUM_FC = 2
    cfg.MODEL.ROI_BOX_HEAD.POOLER_RESOLUTION = 7
    
    cfg.MODEL.ROI_HEADS.IN_FEATURES = ["p2", "p3", "p4", "p5"]
    cfg.MODEL.ANCHOR_GENERATOR.SIZES = [[32], [64], [128], [256], [512]]
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [[0.5, 1.0, 2.0]]
    cfg.MODEL.RPN.IN_FEATURES = ["p2", "p3", "p4", "p5", "p6"]
    cfg.MODEL.RPN.PRE_NMS_TOPK_TRAIN = 2000
    cfg.MODEL.RPN.PRE_NMS_TOPK_TEST = 1000
    cfg.MODEL.RPN.POST_NMS_TOPK_TRAIN = 1000
    cfg.MODEL.RPN.POST_NMS_TOPK_TEST = 1000
    
    cfg.DATASETS.TRAIN = ("my_train_dataset",)
    cfg.DATASETS.TEST = ("my_val_dataset",)
    
    cfg.MODEL.WEIGHTS = "detectron2://COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x/137849600/model_final_f10217.pkl"
    
    cfg.SOLVER.IMS_PER_BATCH = 8
    cfg.SOLVER.BASE_LR = 0.0005
    cfg.SOLVER.MAX_ITER = 4000
    cfg.SOLVER.STEPS = (1000, 2000, 3000)
    cfg.SOLVER.GAMMA = 0.5
    
    cfg.MODEL.ROI_HEADS.NUM_CLASSES = 4
    
    cfg.OUTPUT_DIR = "./output"
    os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
    
    return cfg

def train_model():
    setup_dataset()
    cfg = setup_config()
    trainer = CustomTrainer(cfg)
    trainer.resume_or_load(resume=False)
    trainer.train()
    return cfg, trainer

def evaluate_model(cfg, trainer):
    """학습된 모델 평가"""
    print("Starting evaluation...")
    
    evaluator = COCOEvaluator(
        "my_val_dataset", 
        tasks=("bbox",),
        distributed=False,
        output_dir=os.path.join(cfg.OUTPUT_DIR, "validation")
    )
    
    val_loader = build_detection_test_loader(cfg, "my_val_dataset")
    results = inference_on_dataset(trainer.model, val_loader, evaluator)
    metrics = results['bbox']
    
    print("\nDetailed Evaluation Results:")
    print(f"{'Class':<15} {'Precision':>10} {'Recall':>10} {'mAP50':>10} {'mAP50-95':>10}")
    print("-" * 60)

    classes = MetadataCatalog.get("my_val_dataset").thing_classes
    for i, class_name in enumerate(classes):
        ap = metrics.get(f'AP{i}', 0)
        ap50 = metrics.get(f'AP50{i}', 0)
        
        precision = metrics.get(f'category_precision_{i}', 0)
        if isinstance(precision, list) and len(precision) > 0:
            precision = sum(precision) / len(precision)
            
        recall = metrics.get(f'category_recall_{i}', 0)
        if isinstance(recall, list) and len(recall) > 0:
            recall = sum(recall) / len(recall)
            
        print(f"{class_name:<15} {precision:>10.3f} {recall:>10.3f} {ap50:>10.3f} {ap:>10.3f}")
    
    print("-" * 60)
    # 전체 성능 출력
    mAP = metrics.get('AP', 0)
    mAP50 = metrics.get('AP50', 0)
    avg_precision = metrics.get('precision', 0) if 'precision' in metrics else 0
    avg_recall = metrics.get('recall', 0) if 'recall' in metrics else 0
    
    print(f"{'Total':<15} {avg_precision:>10.3f} {avg_recall:>10.3f} {mAP50:>10.3f} {mAP:>10.3f}")

def main():
    cfg, trainer = train_model()
    print("Training completed!")
    evaluate_model(cfg, trainer)
    print("Evaluation completed!")

if __name__ == "__main__":
    main()

In [ ]:
import os
import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()

from detectron2 import model_zoo
from detectron2.engine import DefaultTrainer, DefaultPredictor
from detectron2.config import get_cfg
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.data.datasets import register_coco_instances
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader
import json

def setup_dataset():
    """데이터셋 등록 - train, val, test"""
    # 기존 데이터셋 등록 제거
    for d in ["my_train_dataset", "my_val_dataset", "my_test_dataset"]:
        if d in DatasetCatalog.list():
            DatasetCatalog.remove(d)
    
    # 클래스 정보 로드
    train_json_file = "C:/Users/AI-LHJ/Desktop/coco-segmentation/train/_annotations.coco.json"
    with open(train_json_file, 'r') as f:
        train_data = json.load(f)
        categories = {cat['id']: cat['name'] for cat in train_data['categories']}
    
    # 데이터셋 새로 등록
    register_coco_instances(
        "my_train_dataset", 
        {}, 
        train_json_file,
        "C:/Users/AI-LHJ/Desktop/coco-segmentation/train"
    )
    
    register_coco_instances(
        "my_val_dataset", 
        {}, 
        "C:/Users/AI-LHJ/Desktop/coco-segmentation/valid/_annotations.coco.json",
        "C:/Users/AI-LHJ/Desktop/coco-segmentation/valid"
    )
    
    # 메타데이터 설정
    metadata = MetadataCatalog.get("my_train_dataset")
    metadata.thing_classes = [categories[i] for i in sorted(categories.keys())]
    MetadataCatalog.get("my_val_dataset").thing_classes = metadata.thing_classes

class CustomTrainer(DefaultTrainer):
    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        if output_folder is None:
            output_folder = os.path.join(cfg.OUTPUT_DIR, "inference")
        os.makedirs(output_folder, exist_ok=True)
        return COCOEvaluator(
            dataset_name, 
            tasks=("bbox", "segm"),  # bbox와 segmentation 모두 평가
            distributed=False, 
            output_dir=output_folder,
            use_fast_impl=False
        )

def setup_config():
    cfg = get_cfg()
    # Mask R-CNN 설정 파일 사용
    cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"))
    
    cfg.DATASETS.TRAIN = ("my_train_dataset",)
    cfg.DATASETS.TEST = ("my_val_dataset",)
    
    # 학습된 Mask R-CNN 가중치 로드
    cfg.MODEL.WEIGHTS = "detectron2://COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x/137849600/model_final_f10217.pkl"
    
    # 학습 파라미터 설정
    cfg.SOLVER.IMS_PER_BATCH = 8
    cfg.SOLVER.BASE_LR = 0.0005
    cfg.SOLVER.MAX_ITER = 4000
    cfg.SOLVER.STEPS = (1000, 2000, 3000)
    cfg.SOLVER.GAMMA = 0.5
    
    # 클래스 수 설정 (4개 클래스)
    cfg.MODEL.ROI_HEADS.NUM_CLASSES = 4
    
    # 출력 디렉토리 설정
    cfg.OUTPUT_DIR = "./output"
    os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
    
    return cfg

def evaluate_model(cfg, trainer):
    """학습된 모델 평가"""
    print("Starting evaluation...")
    
    evaluator = COCOEvaluator(
        "my_val_dataset", 
        tasks=("bbox", "segm"),  # bbox와 segmentation 모두 평가
        distributed=False,
        output_dir=os.path.join(cfg.OUTPUT_DIR, "validation"),
        use_fast_impl=False
    )
    
    val_loader = build_detection_test_loader(cfg, "my_val_dataset")
    results = inference_on_dataset(trainer.model, val_loader, evaluator)
    
    # bbox와 segmentation 결과 모두 출력
    for task in ["bbox", "segm"]:
        if task not in results:
            print(f"No {task} evaluation results found!")
            continue
            
        metrics = results[task]
        
        print(f"\nDetailed {task.upper()} Evaluation Results:")
        print(f"{'Class':<15} {'AP':>10} {'AP50':>10} {'AP75':>10}")
        print("-" * 60)
        
        # 클래스별 성능 출력
        classes = MetadataCatalog.get("my_val_dataset").thing_classes
        for i, class_name in enumerate(classes):
            class_metrics = {}
            for k, v in metrics.items():
                if k.endswith(f'/{i}'):
                    metric_name = k.split('/')[0]
                    class_metrics[metric_name] = v
            
            ap = class_metrics.get('AP', 0)
            ap50 = class_metrics.get('AP50', 0)
            ap75 = class_metrics.get('AP75', 0)
            
            print(f"{class_name:<15} {ap:>10.3f} {ap50:>10.3f} {ap75:>10.3f}")
        
        print("-" * 60)
        # 전체 성능 출력
        print(f"{'Overall':<15} {metrics['AP']:>10.3f} {metrics['AP50']:>10.3f} {metrics['AP75']:>10.3f}")

def train_model():
    setup_dataset()
    cfg = setup_config()
    trainer = CustomTrainer(cfg)
    trainer.resume_or_load(resume=False)
    trainer.train()
    return cfg, trainer

def main():
    cfg, trainer = train_model()
    print("Training completed!")
    evaluate_model(cfg, trainer)
    print("Evaluation completed!")

if __name__ == "__main__":
    main()